## About this report

This report evaluates four retrieval adapters across four agent-memory tasks: finding relevant source files for a code fix, answering API questions from documentation, recalling past architectural decisions, and routing tasks to the correct tool.

Each benchmark uses a real corpus and real ground-truth labels. No synthetic performance numbers appear anywhere in this report.

## How to read this report

- **nDCG@10** (primary metric) — normalized discounted cumulative gain at rank 10; higher is better, max 1.0
- **Recall@k** — fraction of relevant documents found in the top-k results
- **Latency** — wall-clock query time (p50 / p95) after index warm-up

Navigate using the sidebar: one page per benchmark, plus a methodology section.

In [ ]:
import contextlib
import json
import pathlib

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path().resolve()
# When executed by jupyterpress the cwd is the notebook's parent directory.
# Walk up until we find results/.
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists():
        ROOT = _p
        break
RESULTS_DIR = ROOT / 'results'

ADAPTERS = ['sqlite', 'lancedb', 'chromadb', 'tantivy']
ADAPTER_LABELS = {'sqlite': 'SQLite FTS5', 'lancedb': 'LanceDB', 'chromadb': 'ChromaDB', 'tantivy': 'Tantivy'}
BENCHMARKS = ['code-finding', 'doc-search', 'episodic-memory', 'skill-search']
BENCHMARK_LABELS = {'code-finding': 'Code Finding', 'doc-search': 'Doc Search', 'episodic-memory': 'Episodic Memory', 'skill-search': 'Skill Search'}
BENCHMARK_CORPORA = {'code-finding': 'SWE-bench Lite', 'doc-search': 'FastAPI docs', 'episodic-memory': 'Neon ADRs', 'skill-search': 'gorilla-llm/APIBench'}

_OUTER_BG = '#f5f3ef'; _PLOT_BG = '#ffffff'; _INK = '#1a1917'; _MUTED = '#6b6b6b'
_SPINE = '#d8d5d0'; _GRID = '#ebebeb'; _LABEL_CLR = '#7a7370'; _TITLE_CLR = _INK
_ADAPTER_COLORS = {'sqlite': '#bdb9b5', 'lancedb': '#3d8c7a', 'chromadb': '#4b7ebb', 'tantivy': '#d4952a'}
_FALLBACK_COLORS = ['#c96442', '#4b7ebb', '#3d8c7a', '#d4952a', '#bdb9b5']
_TICK_SIZE = 9; _LABEL_SIZE = 8; _TITLE_SIZE = 11.5

mpl.rcParams.update({'figure.dpi': 96, 'font.family': 'sans-serif', 'font.size': _TICK_SIZE,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': _GRID, 'grid.linewidth': 0.7, 'grid.linestyle': '-', 'axes.axisbelow': True})

def _apply_style(fig, ax):
    fig.patch.set_facecolor(_OUTER_BG); ax.set_facecolor(_PLOT_BG)
    ax.spines['left'].set_color(_SPINE); ax.spines['bottom'].set_color(_SPINE)
    ax.spines['left'].set_linewidth(0.7); ax.spines['bottom'].set_linewidth(0.7)
    ax.tick_params(axis='both', colors=_MUTED, labelsize=_TICK_SIZE, length=3, width=0.7)
    ax.xaxis.label.set_color(_MUTED); ax.xaxis.label.set_fontsize(_TICK_SIZE)
    ax.yaxis.label.set_color(_MUTED); ax.yaxis.label.set_fontsize(_TICK_SIZE)

def _bar_color(store, idx):
    return _ADAPTER_COLORS.get(store.lower(), _FALLBACK_COLORS[idx % len(_FALLBACK_COLORS)])

def load_results():
    rows = []
    for f in RESULTS_DIR.glob('**/*.json'):
        with contextlib.suppress(Exception):
            rows.append(json.loads(f.read_text()))
    return pd.DataFrame(rows) if rows else pd.DataFrame()

def best(df, benchmark):
    bdf = df[df['benchmark'] == benchmark]
    return bdf.sort_values('ndcg_at_10', ascending=False).groupby('store').first().reindex(ADAPTERS)

df = load_results()
print(f'Loaded {len(df)} result rows')

In [ ]:
# Key findings table
best_per_bench = []
for b in BENCHMARKS:
    bdf = best(df, b)
    if bdf.empty or bdf['ndcg_at_10'].isna().all():
        continue
    top = bdf['ndcg_at_10'].idxmax()
    best_per_bench.append(
        f'| {BENCHMARK_LABELS[b]} | {BENCHMARK_CORPORA[b]} | **{ADAPTER_LABELS[top]}** | {bdf.loc[top, "ndcg_at_10"]:.3f} |'
    )
header = '| Benchmark | Corpus | Best Adapter | nDCG@10 |\n|-----------|--------|-------------|---------|'
print('## Key Findings\n\n' + header + '\n' + '\n'.join(best_per_bench))

In [ ]:
# Cross-benchmark nDCG@10 comparison
pivot = df.groupby(['benchmark', 'store'])['ndcg_at_10'].max().unstack('store')
pivot = pivot.reindex(columns=ADAPTERS).reindex(BENCHMARKS)
pivot.index = [BENCHMARK_LABELS.get(b, b) for b in pivot.index]

x = np.arange(len(pivot.index))
width = 0.18
fig, ax = plt.subplots(figsize=(8.5, 4))
_apply_style(fig, ax)
for i, adapter in enumerate(ADAPTERS):
    col = pivot[adapter] if adapter in pivot.columns else pd.Series([0] * len(pivot))
    ax.bar(x + i * width, col.fillna(0), width, label=ADAPTER_LABELS[adapter], color=_bar_color(adapter, i))
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(pivot.index)
ax.set_ylabel('nDCG@10')
ax.set_ylim(0, 1.1)
ax.legend(frameon=False, fontsize=_LABEL_SIZE, labelcolor=_MUTED,
          handlelength=1.0, borderpad=0.3, bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_title('nDCG@10 across benchmarks and adapters', color=_TITLE_CLR,
             fontsize=_TITLE_SIZE, fontweight='bold', pad=10)
plt.tight_layout(pad=1.0)
plt.show()

## Benchmark Suite

| Benchmark | Corpus | Task | Queries | Docs |
|-----------|--------|------|---------|------|
| [Code Finding](benchmarks/code-finding) | SWE-bench Lite | Locate files to patch for a GitHub issue | 300 | 215 |
| [Doc Search](benchmarks/doc-search) | FastAPI docs | Answer API questions from documentation | 2 002 | ~890 chunks |
| [Episodic Memory](benchmarks/episodic-memory) | Neon ADRs | Recall past architectural decisions | 1 114 | 247 |
| [Skill Search](benchmarks/skill-search) | gorilla-llm/APIBench | Route a task description to the correct API | 300 | 300 |

## Adapters

| Adapter | Retrieval type | Index | Notes |
|---------|---------------|-------|-------|
| **SQLite FTS5** | BM25 (keyword) | In-memory | No embeddings; OR-mode term matching |
| **LanceDB** | Dense vector (HNSW) | On-disk | Sentence-transformer embeddings |
| **ChromaDB** | Dense vector | Embedded | cosine similarity; fast at small scale |
| **Tantivy** | BM25 (keyword) | On-disk | Rust-native; fast indexing |

All adapters use the same `all-MiniLM-L6-v2` sentence-transformer model for embeddings where applicable.